# 3. Computational cost and hardware

CS 301 · Unit II essentials practice

**Why we do this.** Estimating work and storage helps us choose a feasible computation. Timing small experiments helps us identify what limits its speed.

**By the end, you should be able to:**

- Estimate work and output storage from matrix shapes without allocating large arrays.
- Vectorize a loop and compare equivalent implementations.
- Use measurements to explain Python overhead, memory access, and device-transfer costs.

Estimated working time: **40–50 minutes**, including written responses.

## Working through this notebook

Save your own copy. Run the setup, then work in order. Before each exercise, write a short prediction. Replace `...` in the exercise cells, run the checks, and explain the result in your own words. A passing check verifies the calculation; the written response shows how you interpret it.

Each notebook runs independently. No results from another notebook are needed. The examples use modest arrays; a GPU is optional in Notebook 3.

**Name:** …


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
np.set_printoptions(precision=4, suppress=True)

def ready(*values):
    assert all(v is not Ellipsis and v is not None for v in values), "Replace every ... in the exercise before running its check."

def heatmap(A, title, labels=None, limits=None):
    fig, ax = plt.subplots(figsize=(6, 4))
    scale = {} if limits is None else {"vmin": limits[0], "vmax": limits[1]}
    im = ax.imshow(A, cmap="coolwarm", aspect="auto", **scale)
    ax.set_title(title)
    if labels is not None:
        ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
        ax.set_yticks(range(len(labels)), labels)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    plt.show()

import time
import torch
rng = np.random.default_rng(301)

def measure(f, repeats=3):
    f()  # Untimed warm-up.
    samples = []
    for _ in range(repeats):
        start = time.perf_counter()
        f()
        samples.append(time.perf_counter()-start)
    return float(np.median(samples))

print("GPU available:", torch.cuda.is_available())
print("Timings describe this runtime; expected rankings are not check conditions.")


## 1. Estimate before allocating

Let X have shape `(n, d)` and W shape `(k, d)`. Count the output entries, then multiply by about `2*d` operations for each dot product. Use the leading-order count; omit the bias additions. Output storage is the number of entries times the bytes per value.

Complete `product_costs` to return a dictionary with keys `"XXT"`, `"XTX"`, and `"XWT"`. Each value is `(approximate_operations, output_bytes)`. **Compute only these counts. Do not allocate the products.**


**Predict before running.** For 50,000 word vectors with 300 coordinates each, which product has the larger output: `X @ X.T` or `X.T @ X`? Predict the factors by which work and output storage grow if every dimension doubles.

*Write your prediction here.*


In [ ]:
def product_costs(n, d, k, bytes_per_value=4):
    return {
        "XXT": (..., ...),
        "XTX": (..., ...),
        "XWT": (..., ...),
    }


In [ ]:
small = product_costs(5,3,2,8)
assert set(small) == {"XXT","XTX","XWT"}
for pair in small.values(): ready(*pair)
assert small == {"XXT":(150,200), "XTX":(90,72), "XWT":(60,80)}, "Count output entries, then work per entry."
for n,d,k in [(5,3,2),(9,4,7)]:
    original, doubled = product_costs(n,d,k), product_costs(2*n,2*d,2*k)
    for name in original:
        assert doubled[name][0] == 8*original[name][0]
        assert doubled[name][1] == 4*original[name][1]
rows = [{"product":name, "operations":ops, "output GB (decimal)":size/1e9}
        for name,(ops,size) in product_costs(50000,300,10).items()]
display(pd.DataFrame(rows))
print("Check passed. No large product was allocated.")


**Explain.** Suppose you need neighbours of only one word. What smaller computation would avoid storing all pairwise comparisons? Name two memory costs omitted by the output-storage estimate.

*Write your response here.*


## 2. Vectorize a computation, then measure it

The function below counts days whose high temperature exceeds the previous day's high by more than `step`. These generated temperatures let us vary the array length independently of a data download.

Implement `rises` using slices, a comparison, and a reduction. Use no Python loops or comprehensions. Preserve the meaning of “more than.”


In [ ]:
highs = 60 + np.cumsum(rng.normal(size=100000))

def rises_loops(highs, step):
    count = 0
    for i in range(1, len(highs)):
        if highs[i] - highs[i-1] > step:
            count += 1
    return count


**Predict before running.** Do the loop and vectorized versions have different growth rates in the number of days? Why might their running times differ even if both grow linearly?

*Write your prediction here.*


In [ ]:
def rises(highs, step):
    return ...


In [ ]:
for data, threshold in [(np.array([1,3,2,5]),2), (np.array([8]),1), (np.array([]),1), (highs,.5)]:
    result = rises(data,threshold)
    ready(result)
    assert result == rises_loops(data,threshold), "Compare consecutive days and use a strict threshold."
loop_time = measure(lambda: rises_loops(highs,.5))
array_time = measure(lambda: rises(highs,.5))
display(pd.DataFrame({"method":["Python loop","Array operations"],"seconds":[loop_time,array_time]}))
print("Loop / array time:", loop_time/array_time)
print("Check passed.")


**Explain.** Explain the timing difference using where the loops execute. Why does storing values in a NumPy array alone not remove Python loop overhead?

*Write your response here.*


## 3. Keep the result; change the access order

The matrix below is stored in row-major order: adjacent columns of a row are adjacent in memory. `column_sums_by_columns` visits a column at a time. Write `column_sums_by_rows` to compute the **same column sums**, using one loop over rows and a running vector of sums. Do not transpose or call `X.sum(axis=0)` in your implementation.

A stride gives the byte distance between neighbouring entries along an axis. Cache lines transfer blocks of adjacent data; using nearby entries can reduce memory traffic.


In [ ]:
A = rng.normal(size=(1536,1536))

def column_sums_by_columns(X):
    return np.array([X[:,j].sum() for j in range(X.shape[1])])

print("Shape:", A.shape, "Strides in bytes:", A.strides, "Array MB:", A.nbytes/1e6)


**Predict before running.** Which method reads neighbouring values together? Both compute the same sums: predict whether their arithmetic counts differ.

*Write your prediction here.*


In [ ]:
def column_sums_by_rows(X):
    totals = np.zeros(X.shape[1])
    for row in X:
        ...
    return totals


In [ ]:
for shape in [(5,7),(12,3)]:
    sample = rng.normal(size=shape)
    assert np.allclose(column_sums_by_rows(sample), sample.sum(axis=0)), "Accumulate one total for each column."
methods = [("Read columns",column_sums_by_columns), ("Read rows",column_sums_by_rows), ("NumPy sum",lambda X:X.sum(axis=0))]
layout_times = pd.DataFrame({"method":[name for name,f in methods], "seconds":[measure(lambda f=f:f(A)) for name,f in methods]})
display(layout_times)
print("Check passed.")


**Explain.** Use your timings and `A.strides` to explain the results. Why can NumPy compute column sums efficiently even though a Python loop that separately reads each column is slower on many machines? If your ordering differs, report it and suggest a reason.

*Write your response here.*


## 4. Interpret an array-size experiment

The next cell measures repeated addition on several array sizes. One call performs N additions and accesses two inputs and one output. Complete the calculation of additions per second and the total size of the three arrays. The estimated footprint is **not** a measurement of bytes transferred from main memory: cached values may be reused.


**Predict before running.** Why might repeatedly adding short arrays achieve a different rate from adding arrays too large for the cache? Would one addition reuse a value as much as a matrix product does?

*Write your prediction here.*


In [ ]:
records = []
for N in [4096, 262144, 4194304]:
    a = np.ones(N, dtype=np.float64); other = np.ones_like(a); out = np.empty_like(a)
    repeats = max(3, min(1000, 8000000//N))
    np.add(a,other,out=out)
    samples = []
    for trial in range(3):
        start = time.perf_counter()
        for repeat in range(repeats):
            np.add(a,other,out=out)
        samples.append((time.perf_counter()-start)/repeats)
    records.append((N,float(np.median(samples))))
sizes = np.array([r[0] for r in records])
seconds = np.array([r[1] for r in records])


In [ ]:
additions_per_second = ...
footprint_bytes = ...


In [ ]:
ready(additions_per_second, footprint_bytes)
assert np.allclose(additions_per_second * seconds, sizes), "Rate is work divided by time."
assert np.array_equal(footprint_bytes, sizes * 24), "Count two float64 inputs and one float64 output."
display(pd.DataFrame({"entries":sizes,"three arrays MB":footprint_bytes/1e6,"billion additions/s":additions_per_second/1e9}))
print("Check passed.")


**Explain.** Describe the trend you measured. Explain memory bandwidth and arithmetic throughput as two possible limits. Why can a large matrix product reuse cached values more than an elementwise sum?

*Write your response here.*


## 5. Put tensors on a device

Write `device_layer` to move X, W, and b to the requested device, compute the scores, and return a tensor on that device. It must work on a CPU. If this runtime has a GPU, the same function is also tested there.

The optional GPU benchmark reports both computation on already-resident tensors and the full CPU→GPU→CPU calculation. Its timers synchronize the GPU before stopping. The CPU benchmark includes the computation on CPU tensors. No timing ratio is required to pass.


**Predict before running.** What must be true of the devices holding X, W, and b? Why might copying a small problem to a GPU take longer than computing it on the CPU?

*Write your prediction here.*


In [ ]:
def device_layer(X, W, b, device):
    Xd = ...
    Wd = ...
    bd = ...
    return ...


In [ ]:
torch.manual_seed(301)
Xt = torch.randn(7,4); Wt = torch.randn(3,4); bt = torch.randn(3)
devices = [torch.device("cpu")]
if torch.cuda.is_available(): devices.append(torch.device("cuda"))
for device in devices:
    result = device_layer(Xt,Wt,bt,device)
    ready(result)
    assert result.device.type == device.type, "Return the scores on the requested device."
    assert torch.allclose(result.cpu(),Xt @ Wt.T+bt,atol=1e-5), "Check copies, transpose, and bias."
print("Check passed on:", [d.type for d in devices])


In [ ]:
timings = []
for size in [128,1024]:
    xx=torch.randn(size,size); ww=torch.randn(size,size); bb=torch.randn(size)
    cpu_time=measure(lambda:device_layer(xx,ww,bb,torch.device("cpu")))
    row={"size":size,"CPU seconds":cpu_time}
    if torch.cuda.is_available():
        device=torch.device("cuda")
        xd,wd,bd=xx.to(device),ww.to(device),bb.to(device)
        def gpu_compute():
            torch.cuda.synchronize()
            answer=xd @ wd.T+bd
            torch.cuda.synchronize()
            return answer
        def gpu_full():
            torch.cuda.synchronize()
            answer=device_layer(xx,ww,bb,device).cpu()
            torch.cuda.synchronize()
            return answer
        row["GPU resident seconds"]=measure(gpu_compute)
        row["GPU with transfers seconds"]=measure(gpu_full)
    timings.append(row)
display(pd.DataFrame(timings))
if not torch.cuda.is_available():
    print("No GPU in this runtime. Complete the device exercise and discuss the two timing scopes conceptually.")


**Explain.** Explain the difference between the two GPU timing scopes and why synchronization matters. If a GPU was available, use your measurements; otherwise explain what each timer would include. What changes when the same resident data is used for many computations?

*Write your response here.*


## Final check

Restart the runtime and run your completed notebook from the top. Check that every exercise passes, your figures appear, and every prediction and explanation is filled in. Save the notebook with its outputs.

**Source activities:** Meetings 10–12 timing, operation-count, memory-layout, and PyTorch activities; vectorization homework C4. This practice version reorganizes those activities in the order of the Meeting 15 essentials deck.
